# Project 3: Semantic Segmentation on the Semantic Drone Dataset

**Course:** AI-driven Perception, Learning and Mapping for Drones — Modules 2, 5, 35–39
**Status:** Complete (with documented limitations)

## Summary

A U-Net semantic segmentation model trained on the Semantic Drone Dataset (400 drone images, 23 classes). Three experiments:

1. **Exp 3a:** From-scratch U-Net, CrossEntropy loss → mIoU 0.039
2. **Exp 3b:** From-scratch U-Net, Dice+CE loss → mIoU 0.035
3. **Exp 3d:** Pretrained ResNet-34 encoder, Dice+CE loss → mIoU 0.033

**Key finding:** All three experiments converge to a degenerate solution. The model learns dominant classes (paved-area IoU 0.38, grass 0.21) and fails on the 20 rare classes. The bottleneck is **data distribution**, not model capacity or loss function.

See `README.md` for summary and `project3_writeup.md` for the detailed report.


## Setup

Mount Drive, install dependencies, verify GPU.

In [ ]:
import os, torch
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)

!pip install -q segmentation-models-pytorch albumentations

import segmentation_models_pytorch as smp
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('smp version:', smp.__version__)

## Dataset

Semantic Drone Dataset: 400 drone images at 4000x6000, with pixel-level masks over 23 classes.

**Class imbalance:** Class 1 (paved-area) covers 37.7% of all pixels; 17 classes cover <0.1% each.

In [ ]:
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
from pathlib import Path
import albumentations as A
from albumentations.pytorch import ToTensorV2

class DroneSegDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, crop_size=512, augment=False):
        self.img_paths = sorted(Path(img_dir).glob('*.jpg'))
        self.lbl_dir = Path(lbl_dir)
        transforms = [A.RandomCrop(crop_size, crop_size),
                      A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5)]
        if augment:
            transforms += [A.RandomRotate90(p=0.5), A.RandomBrightnessContrast(p=0.2)]
        transforms += [A.Normalize(mean=(0.485, 0.456, 0.406),
                                   std=(0.229, 0.224, 0.225), max_pixel_value=255.0),
                       ToTensorV2()]
        self.tf = A.Compose(transforms)
    def __len__(self):
        return len(self.img_paths)
    def __getitem__(self, idx):
        img = np.array(Image.open(self.img_paths[idx]).convert('RGB'))
        lbl = np.array(Image.open(self.lbl_dir / (self.img_paths[idx].stem + '.png')))
        out = self.tf(image=img, mask=lbl)
        return out['image'], out['mask'].long()

## U-Net architecture

4-level encoder/decoder with skip connections. 7.8M params from scratch, 24.4M with pretrained ResNet-34.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x): return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size(2) - x1.size(2); diffX = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        return self.conv(torch.cat([x2, x1], dim=1))

class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=23, base_features=32):
        super().__init__()
        f = base_features
        self.inc = DoubleConv(in_channels, f)
        self.down1 = Down(f, f*2); self.down2 = Down(f*2, f*4)
        self.down3 = Down(f*4, f*8); self.down4 = Down(f*8, f*16)
        self.up1 = Up(f*16, f*8); self.up2 = Up(f*8, f*4)
        self.up3 = Up(f*4, f*2); self.up4 = Up(f*2, f)
        self.outc = nn.Conv2d(f, num_classes, kernel_size=1)
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1); x3 = self.down2(x2); x4 = self.down3(x3); x5 = self.down4(x4)
        x = self.up1(x5, x4); x = self.up2(x, x3); x = self.up3(x, x2); x = self.up4(x, x1)
        return self.outc(x)

## Loss functions

CrossEntropy vs Dice+CrossEntropy. Dice loss directly optimizes IoU and up-weights rare classes.

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, logits, targets):
        num_classes = logits.shape[1]
        probs = F.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()
        dice_sum = 0.0; count = 0
        for c in range(num_classes):
            p = probs[:, c]; t = targets_onehot[:, c]
            inter = (p * t).sum(dim=(1, 2))
            union = p.sum(dim=(1, 2)) + t.sum(dim=(1, 2))
            dice = (2 * inter + self.smooth) / (union + self.smooth)
            present = t.sum(dim=(1, 2)) > 0
            if present.any():
                dice_sum += dice[present].mean(); count += 1
        return 1.0 - dice_sum / max(count, 1)

class CombinedLoss(nn.Module):
    def __init__(self, ce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(); self.dice = DiceLoss()
        self.ce_w = ce_weight; self.dice_w = dice_weight
    def forward(self, logits, targets):
        return self.ce_w * self.ce(logits, targets) + self.dice_w * self.dice(logits, targets)

## Training results

| Exp | Loss | Encoder | Val mIoU | Time |
|---|---|---|---|---|
| 3a | CrossEntropy | From scratch | 0.039 | 49 min |
| 3b | Dice+CE | From scratch | 0.035 | 49 min |
| 3d | Dice+CE | Pretrained ResNet-34 | 0.033 | 44 min |

All three converge to a degenerate solution. Training loss decreases, mIoU stays flat.

## Diagnosis

On a validation crop that was 100% class 1, the model predicted 100% class 1 (IoU 0.9997). The model is not broken — it correctly segments dominant classes.

**The bottleneck is data distribution.** Random 512x512 crops on 4000x6000 images sample rare-class instances with <1% probability. Over 15 epochs, the model sees each rare class ~1 time.

See `project3_writeup.md` for the full diagnosis and proposed fixes.